<a href="https://colab.research.google.com/github/gandhias-a11y/colab/blob/main/lab12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 9: Recommendation Systems

In [ ]:
import pandas as pd
import numpy as np
interaction=pd.read_csv('https://bradfordtuckfield.com/purchasehistory1.csv')
interaction.set_index("Unnamed: 0", inplace = True)
print(interaction)

            user1  user2  user3  user4  user5
Unnamed: 0                                   
item1           1      1      0      1      1
item2           1      0      1      1      0
item3           1      1      0      1      1
item4           1      0      1      0      1
item5           1      1      0      0      1


In [ ]:
interaction_withcounts=interaction.copy()
interaction_withcounts.loc[:,'counts']=interaction_withcounts.sum(axis=1)
interaction_withcounts=interaction_withcounts.sort_values(by='counts',ascending=False)
print(list(interaction_withcounts.index))

['item1', 'item3', 'item2', 'item4', 'item5']


In [ ]:
def popularity_based(interaction):
 interaction_withcounts=interaction.copy()
 interaction_withcounts.loc[:,'counts']=interaction_withcounts.sum(axis=1)
 sorted = interaction_withcounts.sort_values(by='counts',ascending=False)
 most_popular=list(sorted.index)
 return(most_popular)

In [ ]:
print(popularity_based(interaction))

['item1', 'item3', 'item2', 'item4', 'item5']


In [ ]:
print(list(interaction.loc['item1',:]))

[1, 1, 0, 1, 1]


In [ ]:
def dot_product(vector1,vector2):
 thedotproduct=np.sum([vector1[k]*vector2[k] for k in range(0,len(vector1))])
 return(thedotproduct)

def vector_norm(vector):
 thenorm=np.sqrt(dot_product(vector,vector))
 return(thenorm)

def cosine_similarity(vector1,vector2):
 thedotproduct=dot_product(vector1,vector2)
 thecosine=thedotproduct/(vector_norm(vector1)*vector_norm(vector2))
 thecosine=np.round(thecosine,4)
 return(thecosine)

In [ ]:
import numpy as np
item1=interaction.loc['item1',:]
item3=interaction.loc['item3',:]
print(cosine_similarity(item1,item3))

1.0


/tmp/ipykernel_6134/1916183493.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  thedotproduct=np.sum([vector1[k]*vector2[k] for k in range(0,len(vector1))])


In [ ]:
item2=list(interaction.loc['item2',:])
item5=list(interaction.loc['item5',:])
print(cosine_similarity(item2,item5))

0.3333


In [ ]:
ouritem='item1'
otherrows=[rowname for rowname in interaction.index if rowname!=ouritem]
otheritems=interaction.loc[otherrows,:]
theitem=interaction.loc[ouritem,:]

In [ ]:
similarities=[]
for items in otheritems.index:
 similarities.append(cosine_similarity(theitem,otheritems.loc[items,:]))

otheritems['similarities']=similarities
recommendations = list(otheritems.sort_values(by='similarities',ascending=False).index)

/tmp/ipykernel_6134/1916183493.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  thedotproduct=np.sum([vector1[k]*vector2[k] for k in range(0,len(vector1))])


In [ ]:
def get_item_recommendations(interaction,itemname):
 otherrows=[rowname for rowname in interaction.index if rowname!=itemname]
 otheritems=interaction.loc[otherrows,:]
 theitem=list(interaction.loc[itemname,:])
 similarities=[]
 for items in otheritems.index:
  similarities.append(cosine_similarity(theitem,list(otheritems.loc[items,:])))
 otheritems['similarities']=similarities
 return list(otheritems.sort_values(by='similarities',ascending=False).index)

In [ ]:
user2=interaction.loc[:,'user2']
user5=interaction.loc[:,'user5']
print(cosine_similarity(user2,user5))

0.866


/tmp/ipykernel_6134/1916183493.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  thedotproduct=np.sum([vector1[k]*vector2[k] for k in range(0,len(vector1))])


In [ ]:
user3=interaction.loc[:,'user3']
user5=interaction.loc[:,'user5']
print(cosine_similarity(user3,user5))

0.3536


/tmp/ipykernel_6134/1916183493.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  thedotproduct=np.sum([vector1[k]*vector2[k] for k in range(0,len(vector1))])


In [ ]:
def get_similar_users(interaction,username):
 othercolumns=[columnname for columnname in interaction.columns if columnname!=username]
 otherusers=interaction[othercolumns]
 theuser=list(interaction[username])
 similarities=[]
 for users in otherusers.columns:
  similarities.append(cosine_similarity(theuser,list(otherusers.loc[:,users])))
 otherusers.loc['similarities',:]=similarities
 return list(otherusers.sort_values(by='similarities',axis=1,ascending=False).columns)

In [ ]:
def get_user_recommendations(interaction,username):
 similar_users=get_similar_users(interaction,username)
 purchase_history=interaction[similar_users[0]]
 purchased=list(purchase_history.loc[purchase_history==1].index)
 purchased2=list(interaction.loc[interaction[username]==1,:].index)
 recs=sorted(list(set(purchased) - set(purchased2)))
 return(recs)

In [ ]:
get_user_recommendations
(interaction,'user2')

(            user1  user2  user3  user4  user5
 Unnamed: 0                                   
 item1           1      1      0      1      1
 item2           1      0      1      1      0
 item3           1      1      0      1      1
 item4           1      0      1      0      1
 item5           1      1      0      0      1,
 'user2')

In [ ]:
import pandas as pd
lastfm = pd.read_csv("https://bradfordtuckfield.com/lastfm-matrix-germany.csv")
print(lastfm.head())

   user  a perfect circle  abba  ac/dc  adam green  aerosmith  afi  air  \
0     1                 0     0      0           0          0    0    0   
1    33                 0     0      0           1          0    0    0   
2    42                 0     0      0           0          0    0    0   
3    51                 0     0      0           0          0    0    0   
4    62                 0     0      0           0          0    0    0   

   alanis morissette  alexisonfire  ...  timbaland  tom waits  tool  \
0                  0             0  ...          0          0     0   
1                  0             0  ...          0          0     0   
2                  0             0  ...          0          0     0   
3                  0             0  ...          0          0     0   
4                  0             0  ...          0          0     0   

   tori amos  travis  trivium  u2  underoath  volbeat  yann tiersen  
0          0       0        0   0          0        

In [ ]:
lastfm.drop(['user'],axis=1,inplace=True)
lastfmt=lastfm.T
print(lastfmt.shape)

(285, 1257)


In [ ]:
get_item_recommendations(lastfmt,'abba')[0:10]

['madonna',
 'robbie williams',
 'elvis presley',
 'michael jackson',
 'queen',
 'the beatles',
 'kelly clarkson',
 'groove coverage',
 'duffy',
 'mika']

In [ ]:
print(get_user_recommendations(lastfmt,0)[0:3])

/tmp/ipykernel_6134/1916183493.py:11: RuntimeWarning: invalid value encountered in scalar divide
  thecosine=thedotproduct/(vector_norm(vector1)*vector_norm(vector2))
/tmp/ipykernel_6134/2817288401.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  otherusers.loc['similarities',:]=similarities


['billy talent', 'bob marley', 'die toten hosen']


Ch 9 Summary:

This chapter covers recommendation systems, algorithms that “analyze data about products and customers to determine which customers would be most receptive to which products”. (191). There are systems that recommend the most popular products to everyone and systems that take into account the customer’s preferences.

The book explains that when a business does not know anything about a customer (known as the cold-start problem), the easiest solution is to recommend the most popular goods. We start off by importing pandas to rearrange the data and view which users bought which item. This binary table is known as an interaction matrix. Through manipulation of a copy of the matrix, we can find and order the most popular items. We consolidate all of this code into a single function (popularity_based()) to end this section.

This next section covers item-based collaborative filtering—the idea that we should recommend items to a customer that are most similar to an item they are already interested in/have already bought. To accomplish this, we have to find a method of quantitatively measuring the relationship between two items so that we can rank items in relation to a chosen one. The method we use is called cosine similarity. We interpret purchase history of an item as a vector and measure the angle between the two vectors (items). The cosine of the angle will range between 0 and 1 and thus can be ranked quantitatively. After writing some code functions to calculate dot products and angle cosines, we can make a for loop and find the cosine for between our chosen item and all other ones. Then we are able to rank the items in alignment with item-based collaborative filtering.

The next section of the chapter focuses on user-based collaborative filtering. Rather than making recommendations based off of the purchase patterns of items, we make the recommendation on the similarity of different customers’ purchase histories. Luckily, we are able to reuse the cosine similarity function from the previous section and input users in instead. We can then create the get_user_recommendations function which creates the ranked list of users in comparison to the user inputted. It will print the items purchased by the most similar user that have not already been purchased by the inputted user. The text remind us that the decision to use user or item based recommendations depends on what business metric we are focusing on.

The author explains that the datasets that we have been using are not very realistic and are very small. So, we pivot to a case study focusing on music recommendations using a much larger dataset from last.fm. We import pandas again to examine the data in the same way we started off the chapter with. First, we use our existing get_item_recommendations function to see what the top recommended musical artists are for fans of ABBA. Then we check out the results for when we use get_user_recommendations. The top three recommended artists with this method are drastically different from our previous output because they are artists our user has not listened to while the earlier output was just what was similar to the user’s history.

The chapter ends by briefly discussing recommendation systems other than collaborative filtering. We can use linear algebra methods, clustering (which we covered in earlier chapters), and content-based comparisons.